# 垃圾分类 GPU 微调：一键运行

点击上方「全部运行」即可自动下载数据、加载上一版 94.62% 的权重并微调。无需手动上传文件或填写 Token。
如果第一步提示没有 GPU：选择「代码执行程序 → 更改运行时类型 → GPU」，然后重新点「全部运行」。
运行结束后会下载结果包，不会提交学校作业。


## 1. 检查 GPU
无需为了本实验更换 Colab 已有的 CUDA/PyTorch 安装。

In [ ]:
import sys
import torch
import torchvision
print('Python:', sys.version.split()[0])
print('PyTorch:', torch.__version__, 'Torchvision:', torchvision.__version__)
assert torch.cuda.is_available(), '没有检测到 CUDA GPU，请先在 Colab 运行时设置中选择 GPU。'
print('GPU:', torch.cuda.get_device_name(0))
print('显存 GiB:', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))


## 2. 自动下载数据和权重
首次运行自动从 GitHub 下载约 30 MB 资源，并校验完整性。


In [ ]:
from pathlib import Path
import hashlib, json, zipfile, stat, time, urllib.request
from concurrent.futures import ThreadPoolExecutor

BASE = 'https://raw.githubusercontent.com/greatzhu-cyber/fuck-ddl/master/garbage-colab/assets'
EXPECTED_SHA256 = '8d13ed892ace3ee4e979da67a298317394b9c88c45d02093547f8d5f4ed2d32a'
archive = Path('/content/garbage_colab_gpu_bundle.zip')
if not archive.exists() or hashlib.sha256(archive.read_bytes()).hexdigest() != EXPECTED_SHA256:
    with urllib.request.urlopen(BASE + '/download_manifest.json', timeout=60) as response:
        download_manifest = json.load(response)
    assert download_manifest['sha256'] == EXPECTED_SHA256, '资源版本不匹配'
    cache = Path('/content/garbage_colab_download_parts')
    cache.mkdir(exist_ok=True)
    def download_part(item):
        assert Path(item['name']).name == item['name']
        target = cache / item['name']
        if target.exists() and hashlib.sha256(target.read_bytes()).hexdigest() == item['sha256']:
            return target
        for attempt in range(3):
            try:
                with urllib.request.urlopen(BASE + '/' + item['name'], timeout=60) as response:
                    content = response.read()
                assert len(content) == item['size']
                assert hashlib.sha256(content).hexdigest() == item['sha256']
                target.write_bytes(content)
                return target
            except Exception:
                if attempt == 2:
                    raise
                time.sleep(attempt + 1)
    print('正在自动下载数据和权重……', flush=True)
    with ThreadPoolExecutor(max_workers=4) as pool:
        downloaded = list(pool.map(download_part, download_manifest['parts']))
    with archive.open('wb') as output:
        for part in downloaded:
            output.write(part.read_bytes())
assert hashlib.sha256(archive.read_bytes()).hexdigest() == EXPECTED_SHA256
WORK = Path('/content/garbage_colab_gpu')
WORK.mkdir(exist_ok=True)
with zipfile.ZipFile(archive) as z:
    for item in z.infolist():
        target = (WORK / item.filename).resolve()
        assert target.is_relative_to(WORK.resolve()), '无效压缩路径'
        assert not stat.S_ISLNK(item.external_attr >> 16), '不接受符号链接'
    z.extractall(WORK)
manifest = json.loads((WORK / 'bundle_manifest.json').read_text())
for name, digest in manifest.items():
    assert hashlib.sha256((WORK / name).read_bytes()).hexdigest() == digest, name
print('数据和权重自动准备完成，无需手动上传。')


## 3. 配置并训练

默认 25 轮、batch size 32、解冻骨干后半部分，自动保留最佳候选。训练前先在 GPU 上复核基线应为 **123/130**。
设置 `SAVE_TO_DRIVE=True` 可把结果逐轮保存到 Google Drive；否则结束后及时下载结果。训练前的候选也会保留，因此验证没有改善时不会强制替换基线。


In [ ]:
EPOCHS = 25
BATCH_SIZE = 32
UNFREEZE_FROM = 10
BACKBONE_LR = 1e-5
HEAD_LR = 1e-4
PATIENCE = 8
SAVE_TO_DRIVE = False

import datetime, subprocess
tag = datetime.datetime.now().strftime('%Y%m%d_%H%M%S_%f')
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT = Path('/content/drive/MyDrive/digit_garbage_gpu_runs') / tag
else:
    OUTPUT = Path('/content/digit_garbage_gpu_runs') / tag
OUTPUT.mkdir(parents=True, exist_ok=False)
command = [sys.executable, str(WORK / 'train_colab_gpu.py'), '--device', 'cuda',
           '--output', str(OUTPUT), '--epochs', str(EPOCHS), '--batch-size', str(BATCH_SIZE),
           '--unfreeze-from', str(UNFREEZE_FROM), '--backbone-lr', str(BACKBONE_LR),
           '--head-lr', str(HEAD_LR), '--patience', str(PATIENCE)]
print('输出目录:', OUTPUT)
with (OUTPUT / 'training.log').open('w', encoding='utf-8') as log:
    process = subprocess.Popen(command, cwd=WORK, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='', flush=True)
        log.write(line)
        log.flush()
    returncode = process.wait()
assert returncode == 0, f'训练中断，日志保存在 {OUTPUT / "training.log"}'


## 4. 查看实际成绩
以下读取本次运行结果；同一验证集用于选模型，不是学校隐藏测试成绩。

In [ ]:
metrics = json.loads((OUTPUT / 'metrics.json').read_text())
assert not metrics['smoke_test'] and metrics['device'] == 'cuda'
print('实际 GPU:', metrics['device_name'])
print(f"基线：{metrics['baseline']['correct']}/130 = {metrics['baseline']['accuracy']:.2%}")
print(f"本次最佳：{metrics['best']['correct']}/130 = {metrics['best']['accuracy']:.2%}")
print('最佳轮次:', metrics['best_epoch'], '（0 表示保留基线）')
print(f"准确率变化：{(metrics['best']['accuracy']-metrics['baseline']['accuracy'])*100:+.2f} 个百分点")
print('保存权重 CPU 重载检查:', metrics['saved_checkpoint_cpu_check'])
print('耗时（分钟）:', round(metrics['elapsed_seconds']/60, 2))


## 5. 下载结果

结果包括权重、日志、逐张验证预测、历史曲线数据及两种框架的预测提交包。
MindSpore 提交包保留原有网络和接口；微调后的 MindSpore 推理仍需学校平台测试。运行本 Notebook 不会提交作业。


In [ ]:
from google.colab import files
# 将最后完整的日志也收入下载包（训练脚本打包时日志仍在写入）。
with zipfile.ZipFile(OUTPUT / 'garbage_gpu_results.zip', 'a', zipfile.ZIP_DEFLATED) as z:
    if 'training.log' not in z.namelist():
        z.write(OUTPUT / 'training.log', 'training.log')
files.download(str(OUTPUT / 'garbage_gpu_results.zip'))


把下载的结果包发回当前对话，即可继续分析错例和平台提交兼容性。GPU 微调未必提高这个小验证集的分数，实际结果由第 4 步读取。

参考：[PyTorch 迁移学习](https://docs.pytorch.org/tutorials/beginner/transfer_learning_tutorial)、[MobileNetV2 源码](https://docs.pytorch.org/vision/main/_modules/torchvision/models/mobilenetv2.html)。
